# Kitchener Water Main Breaks + External Soil Data (Spatial Join Testing) (Anh)

#### Purpose
This notebook explores whether external **soil datasets in Canada** can provide useful soil-related features for our **Kitchener water main break analysis**.

- Loads and reviews the Kitchener **water main breaks** and **water mains** datasets
- Performs basic cleaning and validation
- Converts Kitchener break coordinates from projected `x/y` values into **latitude/longitude**
- Tests **two different external soil datasets** in Canada using spatial joins:
  1. **Ontario Soil Survey Complex** dataset
  2. **CANSIS / Soil Landscapes of Canada (SLC)** dataset
- Checks whether the joined soil attributes are complete, meaningful, and useful for analysis
- Summarises whether soil is suitable as a strong feature for our project

#### Key takeaways
Both spatial joins technically worked, but the resulting soil information was **not strong enough to use as a main factor** in the current analysis:
- The first dataset is dominated by **urban / NA values**, so it gives very limited insight in built-up areas
- The second dataset gives more complete coverage, but is still **too broad/generalised** at this level
- At this stage, soil is better treated as **supporting context or a limitation**, rather than a primary modelling feature

In [10]:
import sys
!{sys.executable} -m pip install geopandas shapely pyproj fiona

In [12]:
#Import required packages
import pandas as pd
import numpy as np
import geopandas as gpd
from pyproj import Transformer

## Convert Kitchener break coordinates

The Kitchener break dataset provides locations as projected `x / y` coordinates.

Because the external soil datasets use geographic coordinate systems (latitude / longitude), we first convert the break locations from:

- **EPSG:26917** (projected UTM coordinates)
- to **EPSG:4326** (standard longitude / latitude)

This allows the break points to be spatially matched against soil polygons in the next steps.

In [16]:
#Load cleaned datasets from previous pre-processing step
breaks_clean = pd.read_csv("../data/processed/kitchener_breaks_clean.csv")
mains_clean = pd.read_csv("../data/processed/kitchener_mains_clean.csv")
pipe_df = pd.read_csv("../data/processed/kitchener_pipe_level.csv")

#Copy dataset
breaks_test = breaks_clean.copy()

#Keep only valid coordinates
breaks_test = breaks_test.dropna(subset=["x", "y"]).copy()

#Create transformer: EPSG:26917 -> EPSG:4326
transformer = Transformer.from_crs("EPSG:26917", "EPSG:4326", always_xy=True)

#Convert x, y to lon, lat
breaks_test["longitude"], breaks_test["latitude"] = transformer.transform(
    breaks_test["x"].values,
    breaks_test["y"].values
)

#Show sample
display(
    breaks_test[[
        "Wat Break Incident ID",
        "Street",
        "x", "y",
        "longitude", "latitude"
    ]].head(10)
)

,Wat Break Incident ID,Street,x,y,longitude,latitude
0,2252,LANCASTER ST W,541741.1448,4.812354e+06,-80.484001,43.462931
1,1311,CLOVERDALE CRES,539253.7576,4.807875e+06,-80.515071,43.422734
2,1325,WREN CRES,545329.4589,4.810392e+06,-80.439808,43.445058
3,1328,GREENBROOK DR,539592.5208,4.808291e+06,-80.510856,43.426469
4,1308,MONTGOMERY RD,543897.8495,4.810175e+06,-80.457516,43.443192
5,1312,WESTHEIGHTS DR,538063.0983,4.808117e+06,-80.529762,43.424980
6,1321,RIVER RD E,547018.7513,4.808296e+06,-80.419113,43.426079
7,1326,BLEAMS RD,538558.0000,4.805205e+06,-80.523854,43.398733
8,2047,MILL ST,541546.9539,4.809431e+06,-80.486624,43.436626
9,1324,KINGSWAY DR,545344.7332,4.808249e+06,-80.439797,43.425757


In [20]:
breaks_test.to_csv("../data/processed/kitchener_breaks_latlon.csv", index=False)

## Test Soil Datasets

### Dataset 1: Ontario Soil Survey Complex

This dataset contains detailed polygon-based soil attributes:
- soil name
- drainage
- texture
- slope
- hydrology-related fields

If the spatial join works and the values are meaningful around Kitchener, this could potentially provide useful soil context for each break record.

In [30]:
soil_gdf = gpd.read_file("../data/Soil_Survey_Complex.geojson")

print(soil_gdf.shape)
print(soil_gdf.columns.tolist())
print(soil_gdf.crs)
print(soil_gdf.head())

(144947, 77)
['OGF_ID', 'MAPUNIT', 'SOIL_CMPLX', 'PERCENT1', 'SOILTYPE1', 'SOILCODE1', 'SOIL_NAME1', 'SYMBOL1', 'PARNT_MAT1', 'LANDSCAPE1', 'SLOPE1', 'CLASS1', 'RANGE1', 'STONINESS1', 'CLI1', 'CLI1_1', 'CLI1_2', 'SURVEY1', 'DRAINAGE1', 'DR_DESIGN1', 'HYDRO1', 'ATEXTURE1', 'MODIFIER1', 'K_FACTOR1', 'PERCENT2', 'SOILTYPE2', 'SOILCODE2', 'SOIL_NAME2', 'SYMBOL2', 'PARNT_MAT2', 'LANDSCAPE2', 'SLOPE2', 'CLASS2', 'RANGE2', 'STONINESS2', 'CLI2', 'CLI2_1', 'CLI2_2', 'SURVEY2', 'DRAINAGE2', 'DR_DESIGN2', 'HYDRO2', 'ATEXTURE2', 'MODIFIER2', 'K_FACTOR2', 'PERCENT3', 'SOILTYPE3', 'SOILCODE3', 'SOIL_NAME3', 'SYMBOL3', 'PARNT_MAT3', 'LANDSCAPE3', 'SLOPE3', 'CLASS3', 'RANGE3', 'STONINESS3', 'CLI3', 'CLI3_1', 'CLI3_2', 'SURVEY3', 'DRAINAGE3', 'DR_DESIGN3', 'HYDRO3', 'ATEXTURE3', 'MODIFIER3', 'K_FACTOR3', 'CLI_SYS', 'ACRES', 'HECTARES', 'LOCATION_ACCURACY', 'GEOMETRY_UPDATE_DATETIME', 'EFFECTIVE_DATETIME', 'SYSTEM_DATETIME', 'OBJECTID', 'SHAPEAREA', 'SHAPELEN', 'geometry']
EPSG:4326
      OGF_ID        

#### Keep only relevant soil fields

The original soil dataset contains many columns, so we keep only the fields most likely to be useful for analysis and interpretation.

In [32]:
soil_keep_cols = [
    "OGF_ID",
    "MAPUNIT",
    "SOIL_CMPLX",
    "PERCENT1",
    "SOIL_NAME1",
    "PARNT_MAT1",
    "SLOPE1",
    "DRAINAGE1",
    "DR_DESIGN1",
    "HYDRO1",
    "ATEXTURE1",
    "K_FACTOR1",
    "geometry"
]

soil_small = soil_gdf[soil_keep_cols].copy()
print(soil_small.shape)
soil_small.head()

(144947, 13)


,OGF_ID,MAPUNIT,SOIL_CMPLX,PERCENT1,SOIL_NAME1,PARNT_MAT1,SLOPE1,DRAINAGE1,DR_DESIGN1,HYDRO1,ATEXTURE1,K_FACTOR1,geometry
0,126293316,ONOND355Ccl/B0,1,100,CLEGG,None,3.5,MW,S1,C,CL,0.040,"POLYGON ((-80.33268 48.43092, -80.33297 48.431..."
1,126293317,ONOND355Shcl/A0,1,100,SHETLAND,None,1.2,P,S1,D,CL,0.040,"POLYGON ((-80.26207 48.43094, -80.26222 48.431..."
2,126293318,ONOND355Bsl=F/A0,2,50,BIZ,None,1.2,P,S3,C,SL,0.017,"POLYGON ((-80.02183 48.42826, -80.02342 48.428..."
3,126293319,ONOND355U/A0,1,100,UNO PARK,None,1.2,VP,S5,D,ORG,0.003,"POLYGON ((-80.40647 48.42367, -80.40658 48.424..."
4,126293310,ONOND355F=Shcl/A0,2,50,FLECK,None,1.2,VP,S5,D,ORG,0.003,"POLYGON ((-80.40051 48.43119, -80.40087 48.430..."


#### Create break point geometry and perform spatial join

We convert the break records into geographic points and then perform a **point-in-polygon spatial join**.

This checks which soil polygon each break location falls inside.

In [34]:
breaks_points = gpd.GeoDataFrame(
    breaks_test.copy(),
    geometry=gpd.points_from_xy(breaks_test["longitude"], breaks_test["latitude"]),
    crs="EPSG:4326"
)

print(breaks_points.shape)
breaks_points.head()

(2994, 21)


,Wat Break Incident ID,Incident date,Type of Asset Broken,Current status of the break,Status last updated date,Nature of Break,Apparent cause of break,Categorization of the Break,Road Segment ID,Closest Civic Number,...,Related Asset ID,Asset Size (cm),Year Asset Installed,Asset Material,Asset Exists,x,y,longitude,latitude,geometry
0,2252,2017-12-01 15:15:00,MAIN,REPAIR COMPLETED,2017-12-02 00:00:00,CORROSION AND FITTING/JOINT,AGE,CATEGORY 2,6961,125,...,134292,450.0,1937.0,CI,Y,541741.1448,4.812354e+06,-80.484001,43.462931,POINT (-80.484 43.46293)
1,1311,2001-03-26 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,6116,76,...,4101323,13.0,1965.0,XXX,Y,539253.7576,4.807875e+06,-80.515071,43.422734,POINT (-80.51507 43.42273)
2,1325,2006-09-06 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,13207,47,...,4099987,25.0,1967.0,XXX,Y,545329.4589,4.810392e+06,-80.439808,43.445058,POINT (-80.43981 43.44506)
3,1328,2006-09-11 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,6498,382,...,4642530,25.0,1964.0,PVC,Y,539592.5208,4.808291e+06,-80.510856,43.426469,POINT (-80.51086 43.42647)
4,1308,2000-01-27 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,22846,224,...,4100648,25.0,1967.0,XXX,Y,543897.8495,4.810175e+06,-80.457516,43.443192,POINT (-80.45752 43.44319)


In [36]:
breaks_with_soil = gpd.sjoin(
    breaks_points,
    soil_small,
    how="left",
    predicate="within"
)

print(breaks_with_soil.shape)
breaks_with_soil.head()

(2994, 34)


,Wat Break Incident ID,Incident date,Type of Asset Broken,Current status of the break,Status last updated date,Nature of Break,Apparent cause of break,Categorization of the Break,Road Segment ID,Closest Civic Number,...,SOIL_CMPLX,PERCENT1,SOIL_NAME1,PARNT_MAT1,SLOPE1,DRAINAGE1,DR_DESIGN1,HYDRO1,ATEXTURE1,K_FACTOR1
0,2252,2017-12-01 15:15:00,MAIN,REPAIR COMPLETED,2017-12-02 00:00:00,CORROSION AND FITTING/JOINT,AGE,CATEGORY 2,6961,125,...,1,100,URBAN,None,-9.0,NA,NA,N,NA,-9.0
1,1311,2001-03-26 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,6116,76,...,1,100,URBAN,None,-9.0,NA,NA,N,NA,-9.0
2,1325,2006-09-06 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,13207,47,...,1,100,URBAN,None,-9.0,NA,NA,N,NA,-9.0
3,1328,2006-09-11 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,6498,382,...,1,100,URBAN,None,-9.0,NA,NA,N,NA,-9.0
4,1308,2000-01-27 00:00:00,SERVICE,REPAIR COMPLETED,2011-12-15 15:38:28,UNKNOWN,OTHER,CATEGORY 1,22846,224,...,1,100,URBAN,None,-9.0,NA,NA,N,NA,-9.0


In [38]:
soil_match_count = breaks_with_soil["SOIL_NAME1"].notna().sum()
soil_total = len(breaks_with_soil)
soil_match_pct = round(soil_match_count / soil_total * 100, 2)

print("Break rows with soil match:", soil_match_count)
print("Total break rows:", soil_total)
print("Soil match %:", soil_match_pct)

Break rows with soil match: 2994
Total break rows: 2994
Soil match %: 100.0


#### Spatial join result

The spatial join completed successfully and all **2,994 break records** matched to a soil polygon.

At first glance, this looks promising because there are no missing joins.  
However, the real question is whether the returned soil values are actually informative for urban Kitchener.

#### Inspect the returned soil values

Next step, we check the main soil fields to see whether they contain:
- meaningful variation
- realistic soil categories
- enough usable detail for analysis

In [40]:
#Inspect exact raw values with repr(), so spaces/newlines become visible
sample_vals = breaks_with_soil["ATEXTURE1"].dropna().astype(str).unique()[:50]

for v in sample_vals:
    print(repr(v))

'NA'
'FSL'
'GL'
'L'
'SIL'
'SL'


In [42]:
#Clean ATEXTURE1
breaks_with_soil["ATEXTURE1_CLEAN"] = (
    breaks_with_soil["ATEXTURE1"]
    .astype(str)
    .str.strip()
    .str.upper()
)

#Turn obvious fake values back to missing
breaks_with_soil["ATEXTURE1_CLEAN"] = breaks_with_soil["ATEXTURE1_CLEAN"].replace({
    "NAN": pd.NA,
    "": pd.NA
})

print(breaks_with_soil["ATEXTURE1_CLEAN"].value_counts(dropna=False).head(30))

ATEXTURE1_CLEAN
NA     2950
FSL      23
SL       11
GL        7
L         2
SIL       1
Name: count, dtype: int64


In [44]:
for col in ["SOIL_NAME1", "DRAINAGE1", "HYDRO1", "PARNT_MAT1", "ATEXTURE1_CLEAN"]:
    print(f"\n=== {col} ===")
    print(breaks_with_soil[col].isna().mean() * 100)
    print(breaks_with_soil[col].value_counts(dropna=False).head(20))


=== SOIL_NAME1 ===
0.0
SOIL_NAME1
URBAN                       2943
WATERLOO FINE SANDY LOAM      23
BURFORD GRAVELLY LOAM          7
LISBON SANDY LOAM              7
WATER                          4
BUILT UP AREA                  3
ELMIRA LOAM                    2
FOX SANDY LOAM                 2
ST CLEMENTS SILT LOAM          1
CALEDON SANDY LOAM             1
BOOKTON SANDY LOAM             1
Name: count, dtype: int64

=== DRAINAGE1 ===
0.0
DRAINAGE1
NA    2946
W       42
WA       4
P        2
Name: count, dtype: int64

=== HYDRO1 ===
0.0
HYDRO1
N    2950
A      40
C       3
B       1
Name: count, dtype: int64

=== PARNT_MAT1 ===
100.0
PARNT_MAT1
None    2994
Name: count, dtype: int64

=== ATEXTURE1_CLEAN ===
0.0
ATEXTURE1_CLEAN
NA     2950
FSL      23
SL       11
GL        7
L         2
SIL       1
Name: count, dtype: int64


#### Key findings

Main issue:
- `SOIL_NAME1` is dominated by **URBAN** for almost all break records
- Many other fields are mostly coded as **NA / placeholder values**
- Soil texture values show very little variation

This means the dataset is technically matching the locations, but in a built-up urban area like Kitchener, it is mostly returning broad urban classifications rather than meaningful subsurface soil detail.

In [46]:
#Main soil texture distribution from break-level enriched dataset
texture_counts = breaks_with_soil["ATEXTURE1"].value_counts(dropna=False)

print("Top soil texture categories in break records:")
display(texture_counts.head(20))

#Percentage version
texture_pct = (
    breaks_with_soil["ATEXTURE1"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("Top soil texture categories (%) in break records:")
display(texture_pct.head(20))

Top soil texture categories in break records:


ATEXTURE1
NA     2950
FSL      23
SL       11
GL        7
L         2
SIL       1
Name: count, dtype: int64

Top soil texture categories (%) in break records:


ATEXTURE1
NA     98.53
FSL     0.77
SL      0.37
GL      0.23
L       0.07
SIL     0.03
Name: proportion, dtype: float64

#### Simplify soil texture categories

To make the texture values easier to interpret, we group detailed texture codes into broader categories such as:
- Sand-heavy
- Loam / mixed
- Silt-heavy
- Other / mixed

This is mainly for readability and to confirm whether any useful pattern exists.

In [48]:
def simplify_texture(val):
    if pd.isna(val):
        return "Unknown"
    
    v = str(val).strip().upper()

    # Organic
    if v in ["ORG", "PEAT", "PT", "MUCK", "M", "O"]:
        return "Organic / Peat"

    # Sand-dominant textures
    elif v in ["S", "FS", "LS", "SL", "CSL", "FSL", "LFS", "LVFS", "VFSL", "GS", "GSL"]:
        return "Sand-heavy"

    # Clay-dominant textures
    elif v in ["C", "CL", "SIC", "SICL"]:
        return "Clay-heavy"

    # Loam textures
    elif v in ["L", "GL"]:
        return "Loam / mixed"

    # Silt-dominant textures
    elif v in ["SIL"]:
        return "Silt-heavy"

    # Gravel / variable / other special
    elif v in ["GRAV", "VAR", "R"]:
        return "Other / Mixed"

    else:
        return "Other / Mixed"

breaks_with_soil["ATEXTURE1_CLEAN"] = (
    breaks_with_soil["ATEXTURE1"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"NAN": pd.NA, "": pd.NA})
)

breaks_with_soil["SOIL_TEXTURE_GROUP"] = breaks_with_soil["ATEXTURE1_CLEAN"].apply(simplify_texture)

group_counts = breaks_with_soil["SOIL_TEXTURE_GROUP"].value_counts(dropna=False)
group_pct = breaks_with_soil["SOIL_TEXTURE_GROUP"].value_counts(normalize=True, dropna=False).mul(100).round(2)

print("Broad soil groups in break records:")
display(group_counts)

print("Broad soil groups in break records (%):")
display(group_pct)

Broad soil groups in break records:


SOIL_TEXTURE_GROUP
Other / Mixed    2950
Sand-heavy         34
Loam / mixed        9
Silt-heavy          1
Name: count, dtype: int64

Broad soil groups in break records (%):


SOIL_TEXTURE_GROUP
Other / Mixed    98.53
Sand-heavy        1.14
Loam / mixed      0.30
Silt-heavy        0.03
Name: proportion, dtype: float64

#### Insights

Many values are dominated by broad urban / missing / generic categories, which limits their usefulness as explanatory variables for pipe failure modelling.

Even after simplifying the texture values, the result is still dominated by a single broad category and provides very limited analytical value.

Therefore, Dataset 1 is **not strong enough** to use as a meaningful soil feature source for the Kitchener break analysis.

### Dataset 2: CANSIS / Soil Landscapes of Canada (SLC)

Because the first soil dataset was not useful enough in urban areas, we tested a second national-level source from **CANSIS / Soil Landscapes of Canada (SLC)**.

This dataset is more structured and can be linked in stages:

1. **SLC polygon layer** gives the landscape polygon (`POLY_ID`)
2. **CMP table** gives the dominant soil component for each polygon
3. **Soil name table (SNT)** adds interpretable soil attributes such as:
   - soil name
   - drainage
   - texture
   - soil group


In [50]:
#CANSIS dataset
#File paths
slc_shp_path = "../data/ca_all_slc_v3r2.shp"
cmp_path = "../data/ca_all_slc_v3r2_cmp.dbf"
snt_path = "../data/soil_name_canada_v2r20231107.dbf" 

#Load SLC polygon layer
slc_gdf = gpd.read_file(slc_shp_path)

print("SLC polygons shape:", slc_gdf.shape)
print("SLC CRS:", slc_gdf.crs)
print("SLC columns:")
print(slc_gdf.columns.tolist())

#Load CMP table
cmp_df = pd.read_csv(cmp_path) if cmp_path.endswith(".csv") else gpd.read_file(cmp_path)

#If gpd.read_file on DBF returns a GeoDataFrame-like table, convert to plain DataFrame
cmp_df = pd.DataFrame(cmp_df)

print("\nCMP shape:", cmp_df.shape)
print("CMP columns:")
print(cmp_df.columns.tolist())

#Load Soil Name Table (optional)
try:
    snt_df = gpd.read_file(snt_path)
    snt_df = pd.DataFrame(snt_df)
    print("\nSNT shape:", snt_df.shape)
    print("SNT columns:")
    print(snt_df.columns.tolist())
except Exception as e:
    snt_df = None
    print("\nCould not load Soil Name Table yet:", e)

SLC polygons shape: (12353, 5)
SLC CRS: EPSG:4269
SLC columns:
['AREA', 'PERIMETER', 'POLY_ID', 'ECO_ID', 'geometry']

CMP shape: (20425, 12)
CMP columns:
['POLY_ID', 'CMP', 'PERCENT', 'SLOPE', 'STONE', 'LOCSF', 'PROVINCE', 'SOIL_CODE', 'MODIFIER', 'PROFILE', 'SOIL_ID', 'CMP_ID']

SNT shape: (14581, 26)
SNT columns:
['SOIL_ID', 'PROVINCE', 'SOIL_CODE', 'MODIFIER', 'PROFILE', 'SOILNAME', 'KIND', 'WATERTBL', 'ROOTRESTRI', 'RESTR_TYPE', 'DRAINAGE', 'PMTEX1', 'PMTEX2', 'PMTEX3', 'PMCHEM1', 'PMCHEM2', 'PMCHEM3', 'MDEP1', 'MDEP2', 'MDEP3', 'ORDER2', 'G_GROUP2', 'S_GROUP2', 'ORDER3', 'G_GROUP3', 'S_GROUP3']


#### Limit the national dataset to the Kitchener-Waterloo area

The CANSIS dataset is national in scale, so we first narrow it down to an approximate Kitchener-Waterloo bounding box to keep the spatial join focused and efficient.

In [52]:
#Approximate Kitchener-Waterloo bounding box in EPSG:4326
min_lon, max_lon = -80.65, -80.30
min_lat, max_lat = 43.35, 43.55

#Clip SLC polygons to Kitchener-Waterloo area
slc_kw = slc_gdf.cx[min_lon:max_lon, min_lat:max_lat].copy()

print("Kitchener-Waterloo SLC polygons:", slc_kw.shape)
display(slc_kw.head())

Kitchener-Waterloo SLC polygons: (7, 5)


,AREA,PERIMETER,POLY_ID,ECO_ID,geometry
12242,0.095838,2.794653,560003,560,"POLYGON ((-80.05926 43.76209, -80.0595 43.7609..."
12250,0.219033,4.454710,557005,557,"POLYGON ((-80.65324 43.607, -80.65165 43.60635..."
12251,0.062583,2.943434,560004,560,"POLYGON ((-80.14708 43.57857, -80.14679 43.577..."
12255,0.064518,1.731529,564005,564,"POLYGON ((-80.13199 43.56943, -80.12815 43.569..."
12258,0.015487,0.760011,560005,560,"POLYGON ((-80.6308 43.57209, -80.63043 43.5720..."


#### Select the dominant soil component per polygon

Each SLC polygon can contain multiple soil components.

To simplify the analysis, we keep only the **dominant component** for each polygon, based on the highest percentage value.

This gives one main soil interpretation per polygon.

In [54]:
#Keep useful CMP fields only
cmp_keep_cols = [
    "POLY_ID",
    "CMP",
    "PERCENT",
    "SLOPE",
    "STONE",
    "LOCSF",
    "PROVINCE",
    "SOIL_CODE",
    "MODIFIER",
    "PROFILE",
    "SOIL_ID",
    "CMP_ID"
]

cmp_small = cmp_df[[c for c in cmp_keep_cols if c in cmp_df.columns]].copy()

print("CMP small shape:", cmp_small.shape)
display(cmp_small.head())

#Ensure PERCENT is numeric
cmp_small["PERCENT"] = pd.to_numeric(cmp_small["PERCENT"], errors="coerce")

#Keep dominant component per POLY_ID
cmp_dominant = (
    cmp_small
    .sort_values(["POLY_ID", "PERCENT"], ascending=[True, False])
    .drop_duplicates(subset=["POLY_ID"], keep="first")
    .copy()
)

print("Dominant CMP shape:", cmp_dominant.shape)
display(cmp_dominant.head())

CMP small shape: (20425, 12)


,POLY_ID,CMP,PERCENT,SLOPE,STONE,LOCSF,PROVINCE,SOIL_CODE,MODIFIER,PROFILE,SOIL_ID,CMP_ID
0,242021,1,40,A,U,D,AB,BUF,gl~~~,N,ABBUFgl~~~N,24202101
1,242021,2,30,A,N,U,AB,KEG,~~~~~,N,ABKEG~~~~~N,24202102
2,242021,3,30,A,N,F13,AB,MLD,~~~~~,N,ABMLD~~~~~N,24202103
3,242022,1,40,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202201
4,242022,2,30,A,N,F13,AB,MLD,~~~~~,N,ABMLD~~~~~N,24202202


Dominant CMP shape: (4579, 12)


,POLY_ID,CMP,PERCENT,SLOPE,STONE,LOCSF,PROVINCE,SOIL_CODE,MODIFIER,PROFILE,SOIL_ID,CMP_ID
0,242021,1,40,A,U,D,AB,BUF,gl~~~,N,ABBUFgl~~~N,24202101
3,242022,1,40,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202201
6,242023,1,40,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202301
9,242024,1,60,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202401
12,242025,1,30,A,S,U,AB,COS,~~~~~,A,ABCOS~~~~~A,24202501


In [56]:
#breaks_test should already contain longitude / latitude from earlier conversion
breaks_points = gpd.GeoDataFrame(
    breaks_test.copy(),
    geometry=gpd.points_from_xy(breaks_test["longitude"], breaks_test["latitude"]),
    crs="EPSG:4326"
)

print("Break points shape:", breaks_points.shape)
display(breaks_points[["Wat Break Incident ID", "Street", "longitude", "latitude"]].head())

Break points shape: (2994, 21)


,Wat Break Incident ID,Street,longitude,latitude
0,2252,LANCASTER ST W,-80.484001,43.462931
1,1311,CLOVERDALE CRES,-80.515071,43.422734
2,1325,WREN CRES,-80.439808,43.445058
3,1328,GREENBROOK DR,-80.510856,43.426469
4,1308,MONTGOMERY RD,-80.457516,43.443192


#### Spatially join break points to SLC polygons

We then spatially match each break point to its SLC polygon.

Before joining, the polygon layer is reprojected to the same coordinate reference system as the break points to ensure the spatial join is valid.

In [58]:
#Keep only minimal polygon fields first
slc_poly_keep = ["POLY_ID", "ECO_ID", "geometry"]
slc_kw_small = slc_kw[[c for c in slc_poly_keep if c in slc_kw.columns]].copy()

#Spatial join
breaks_slc = gpd.sjoin(
    breaks_points,
    slc_kw_small,
    how="left",
    predicate="within"
)

print("Breaks joined to SLC polygons:", breaks_slc.shape)
print("Break rows with POLY_ID:", breaks_slc["POLY_ID"].notna().sum())
print("Break rows without POLY_ID:", breaks_slc["POLY_ID"].isna().sum())

display(breaks_slc[["Wat Break Incident ID", "Street", "POLY_ID", "ECO_ID"]].head())

Breaks joined to SLC polygons: (2994, 24)
Break rows with POLY_ID: 2994
Break rows without POLY_ID: 0


/var/folders/bk/sln2b49n47g24cwmlq18xqhc0000gn/T/ipykernel_47211/3793365015.py:6: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  breaks_slc = gpd.sjoin(


,Wat Break Incident ID,Street,POLY_ID,ECO_ID
0,2252,LANCASTER ST W,560004,560
1,1311,CLOVERDALE CRES,560005,560
2,1325,WREN CRES,560006,560
3,1328,GREENBROOK DR,560005,560
4,1308,MONTGOMERY RD,560006,560


In [60]:
#Fix CRS mismatch before spatial join
#breaks_points = EPSG:4326
#slc_kw_small = EPSG:4269
#Reproject SLC polygons to match breaks
slc_kw_small = slc_kw_small.to_crs(breaks_points.crs)

print("Break CRS:", breaks_points.crs)
print("SLC CRS after reprojection:", slc_kw_small.crs)

#Spatial join again after CRS fix
breaks_slc = gpd.sjoin(
    breaks_points,
    slc_kw_small,
    how="left",
    predicate="within"
)

print("Breaks joined to SLC polygons:", breaks_slc.shape)
print("Break rows with POLY_ID:", breaks_slc["POLY_ID"].notna().sum())
print("Break rows without POLY_ID:", breaks_slc["POLY_ID"].isna().sum())

display(breaks_slc[["Wat Break Incident ID", "Street", "POLY_ID", "ECO_ID"]].head())

Break CRS: EPSG:4326
SLC CRS after reprojection: EPSG:4326
Breaks joined to SLC polygons: (2994, 24)
Break rows with POLY_ID: 2994
Break rows without POLY_ID: 0


,Wat Break Incident ID,Street,POLY_ID,ECO_ID
0,2252,LANCASTER ST W,560004,560
1,1311,CLOVERDALE CRES,560005,560
2,1325,WREN CRES,560006,560
3,1328,GREENBROOK DR,560005,560
4,1308,MONTGOMERY RD,560006,560


In [62]:
#Keep only useful CMP columns
cmp_keep_cols = [
    "POLY_ID",
    "CMP",
    "PERCENT",
    "SLOPE",
    "STONE",
    "LOCSF",
    "PROVINCE",
    "SOIL_CODE",
    "MODIFIER",
    "PROFILE",
    "SOIL_ID",
    "CMP_ID"
]

cmp_small = cmp_df[[c for c in cmp_keep_cols if c in cmp_df.columns]].copy()

#Make sure PERCENT is numeric
cmp_small["PERCENT"] = pd.to_numeric(cmp_small["PERCENT"], errors="coerce")

# Keep dominant component only
# For each POLY_ID, keep the row with highest PERCENT
cmp_dominant = (
    cmp_small
    .sort_values(["POLY_ID", "PERCENT"], ascending=[True, False])
    .drop_duplicates(subset=["POLY_ID"], keep="first")
    .copy()
)

print("Dominant CMP shape:", cmp_dominant.shape)
display(cmp_dominant.head())

Dominant CMP shape: (4579, 12)


,POLY_ID,CMP,PERCENT,SLOPE,STONE,LOCSF,PROVINCE,SOIL_CODE,MODIFIER,PROFILE,SOIL_ID,CMP_ID
0,242021,1,40,A,U,D,AB,BUF,gl~~~,N,ABBUFgl~~~N,24202101
3,242022,1,40,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202201
6,242023,1,40,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202301
9,242024,1,60,A,N,B07,AB,MKW,~~~~~,N,ABMKW~~~~~N,24202401
12,242025,1,30,A,S,U,AB,COS,~~~~~,A,ABCOS~~~~~A,24202501


In [64]:
#Attach dominant CMP info using POLY_ID
breaks_slc_cmp = breaks_slc.merge(
    cmp_dominant,
    on="POLY_ID",
    how="left"
)

print("Breaks + dominant CMP shape:", breaks_slc_cmp.shape)

display(
    breaks_slc_cmp[
        ["Wat Break Incident ID", "Street", "POLY_ID", "SOIL_ID", "SOIL_CODE", "PERCENT", "SLOPE", "STONE"]
    ].head()
)

Breaks + dominant CMP shape: (2994, 35)


,Wat Break Incident ID,Street,POLY_ID,SOIL_ID,SOIL_CODE,PERCENT,SLOPE,STONE
0,2252,LANCASTER ST W,560004,ONBUF~~~~~A,BUF,37,B,S
1,1311,CLOVERDALE CRES,560005,ONWTO~~~~~A,WTO,29,B,N
2,1325,WREN CRES,560006,ONFOX~~~~~A,FOX,20,B,N
3,1328,GREENBROOK DR,560005,ONWTO~~~~~A,WTO,29,B,N
4,1308,MONTGOMERY RD,560006,ONFOX~~~~~A,FOX,20,B,N


In [66]:
#Keep only useful columns from soil name table
snt_keep_cols = [
    "SOIL_ID",
    "SOILNAME",
    "DRAINAGE",
    "PMTEX1",
    "PMTEX2",
    "PMTEX3",
    "ORDER3",
    "G_GROUP3",
    "S_GROUP3"
]

snt_small = snt_df[[c for c in snt_keep_cols if c in snt_df.columns]].copy()

#Merge SNT onto break-level data using SOIL_ID
breaks_slc_cmp_named = breaks_slc_cmp.merge(
    snt_small,
    on="SOIL_ID",
    how="left"
)

print("Breaks + CMP + SNT shape:", breaks_slc_cmp_named.shape)

display(
    breaks_slc_cmp_named[
        ["SOIL_ID", "SOIL_CODE", "SOILNAME", "DRAINAGE", "PMTEX1", "ORDER3", "G_GROUP3", "S_GROUP3"]
    ].drop_duplicates().head(20)
)

Breaks + CMP + SNT shape: (2994, 43)


,SOIL_ID,SOIL_CODE,SOILNAME,DRAINAGE,PMTEX1,ORDER3,G_GROUP3,S_GROUP3
0,ONBUF~~~~~A,BUF,BURFORD,W,VC,LU,GBL,O
1,ONWTO~~~~~A,WTO,WATERLOO,W,VC,LU,GBL,BR
2,ONFOX~~~~~A,FOX,FOX,W,VC,LU,GBL,BR
2558,ONGUP~~~~~A,GUP,GUELPH,W,M,LU,GBL,BR


#### Spatial join result

The second spatial join also completed successfully:

- All **2,994 break records** matched to an SLC polygon
- No break records were left unmatched

This confirms the geographic coverage is sufficient for the Kitchener area.

In [68]:
#Check missingness of key soil fields
soil_check_cols = [
    "POLY_ID",
    "SOIL_ID",
    "SOIL_CODE",
    "SOILNAME",
    "DRAINAGE",
    "PMTEX1",
    "SLOPE",
    "STONE"
]

soil_missing_summary = pd.DataFrame({
    "missing_count": breaks_slc_cmp_named[soil_check_cols].isnull().sum(),
    "missing_ratio_pct": (breaks_slc_cmp_named[soil_check_cols].isnull().mean() * 100).round(2)
}).sort_values("missing_ratio_pct", ascending=False)

display(soil_missing_summary)

,missing_count,missing_ratio_pct
POLY_ID,0,0.0
SOIL_ID,0,0.0
SOIL_CODE,0,0.0
SOILNAME,0,0.0
DRAINAGE,0,0.0
PMTEX1,0,0.0
SLOPE,0,0.0
STONE,0,0.0


#### Add dominant soil component and readable soil attributes

In [70]:
for col in ["SOIL_CODE", "SOILNAME", "DRAINAGE", "PMTEX1", "SLOPE", "STONE"]:
    print(f"\n=== {col} ===")
    print(breaks_slc_cmp_named[col].value_counts(dropna=False).head(20))


=== SOIL_CODE ===
SOIL_CODE
FOX    1718
BUF     701
WTO     572
GUP       3
Name: count, dtype: int64

=== SOILNAME ===
SOILNAME
FOX         1718
BURFORD      701
WATERLOO     572
GUELPH         3
Name: count, dtype: int64

=== DRAINAGE ===
DRAINAGE
W    2994
Name: count, dtype: int64

=== PMTEX1 ===
PMTEX1
VC    2991
M        3
Name: count, dtype: int64

=== SLOPE ===
SLOPE
B    2994
Name: count, dtype: int64

=== STONE ===
STONE
N    2290
S     704
Name: count, dtype: int64


In [72]:
def texture_group(val):
    if pd.isna(val):
        return "Unknown"
    
    v = str(val).strip().upper()

    # Coarse / sandy / variable coarse
    if v in ["S", "FS", "LS", "LFS", "LVFS", "GS", "VC"]:
        return "Sand-dominant"
    
    # Loam / sandy loam / mixed loam
    elif v in ["L", "SL", "FSL", "VFSL", "GL", "GSL"]:
        return "Loam / sandy-loam"
    
    # Fine / clay / silt / clay-loam style
    elif v in ["CL", "C", "SIL", "SICL", "SIC", "CSL", "M"]:
        return "Silt / clay-dominant"
    
    else:
        return "Other / Mixed"

breaks_slc_cmp_named["TEXTURE_GROUP"] = breaks_slc_cmp_named["PMTEX1"].apply(texture_group)

print("\n=== TEXTURE_GROUP ===")
print(breaks_slc_cmp_named["TEXTURE_GROUP"].value_counts(dropna=False))
print(
    breaks_slc_cmp_named["TEXTURE_GROUP"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


=== TEXTURE_GROUP ===
TEXTURE_GROUP
Sand-dominant           2991
Silt / clay-dominant       3
Name: count, dtype: int64
TEXTURE_GROUP
Sand-dominant           99.9
Silt / clay-dominant     0.1
Name: proportion, dtype: float64


#### Insights

Although Dataset 2 is much more complete than Dataset 1, it still appears **too broad and generalised** for this project.

Main observations:
- Most break records fall into only a small number of broad soil classes (mainly **FOX**, **BURFORD**, and **WATERLOO**)
- Drainage is effectively constant (`W`) across all break records
- Texture is almost entirely one dominant class (`VC`), which becomes **Sand-dominant** after grouping

This means the dataset provides cleaner coverage, but still offers **very limited variation** across the break locations.

## Conclusion

Both external soil datasets were successfully joined to the Kitchener break locations, so the spatial workflow itself is valid.

However, the returned soil information was not strong enough to support pipe-level failure modelling as a primary feature:
- Dataset 1 was too weak in urban areas
- Dataset 2 was more complete, but still too broad/generalised

At this stage, soil is better treated as a project limitation, or a future improvement if more pipe-specific geotechnical data becomes available.

For the main modelling stage, stronger variables such as pipe age, material, size, condition, and break history remain the better focus.